# ⚙️ Notebook 03 — Sliding Window + Feature Engineering

**Input:** `data/synthetic/helmet_imu_raw.csv`  
**Output:** `data/processed/features.csv` — one row per window, 50+ features  

---

## Why Sliding Windows?

Raw IMU gives 6 numbers per timestep. That's hard to classify directly.  
Instead, we take a **window of 20 timesteps** and compute statistics over it:

```
[t0...t19]  →  mean_ax, std_ax, max_az, jerk_mean, tilt_angle, ...  →  label
[t10...t29] →  mean_ax, std_ax, ...                                  →  label
[t20...t39] →  ...                                                    →  label
```

This converts time-series → tabular ML problem.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import apply_sliding_window, split_dataset, WINDOW_SIZE, STRIDE

df = pd.read_csv('../data/synthetic/helmet_imu_raw.csv')
print(f'Raw shape: {df.shape}')

In [ ]:
# ── Apply sliding window + extract features ──────────────────────
print(f'Window size: {WINDOW_SIZE}, Stride: {STRIDE}')
print('Extracting features...\n')

feature_df = apply_sliding_window(df, window_size=WINDOW_SIZE, stride=STRIDE, verbose=True)

# Save
os.makedirs('../data/processed', exist_ok=True)
feature_df.to_csv('../data/processed/features.csv', index=False)

print(f'\nFeature matrix shape: {feature_df.shape}')
display(feature_df.head(3))

In [ ]:
# ── List all features ────────────────────────────────────────────
feature_cols = [c for c in feature_df.columns if c not in ('label', 'session_id')]
print(f'Total features: {len(feature_cols)}')
print('\nFeature groups:')
groups = {}
for f in feature_cols:
    prefix = f.split('_')[0] if '_' in f else 'other'
    groups.setdefault(prefix, []).append(f)
for grp, feats in groups.items():
    print(f'  {grp:>12}: {feats}')

In [ ]:
# ── Feature correlation with label ───────────────────────────────
corr_with_label = feature_df[feature_cols + ['label']].corr()['label'].drop('label')
top20 = corr_with_label.abs().nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#1a1a2e')

colors = ['#e74c3c' if corr_with_label[f] < 0 else '#4facfe' for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1], color=colors[::-1], alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], color='#e0e0e0', fontsize=8)
ax.set_xlabel('|Correlation with label|', color='#aaaaaa')
ax.set_title('Top 20 Features — Correlation with Label', color='white', fontsize=12)
ax.tick_params(colors='#aaaaaa')
ax.grid(True, axis='x', color='#222244', linewidth=0.5)
for spine in ax.spines.values(): spine.set_edgecolor('#333355')

plt.tight_layout()
plt.show()
print('\nTop predictive features:')
print(corr_with_label.abs().nlargest(10))

In [ ]:
# ── Feature distributions — key features by class ────────────────
key_features = ['jerk_max', 'gyro_mag_max', 'accel_mag_max', 'tilt_max_deg', 'gz_range']
COLORS = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}
NAMES  = {0: 'Normal', 1: 'Pothole', 2: 'SuddenBrake', 3: 'Crash'}

fig, axes = plt.subplots(1, len(key_features), figsize=(16, 5))
fig.patch.set_facecolor('#0d0d0d')

for i, feat in enumerate(key_features):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    for lbl in range(4):
        data = feature_df[feature_df['label'] == lbl][feat]
        ax.hist(data, bins=40, alpha=0.6, color=COLORS[lbl], density=True)
    ax.set_title(feat.replace('_', '\n'), color='#e0e0e0', fontsize=9)
    ax.tick_params(colors='#aaaaaa', labelsize=7)
    ax.grid(True, color='#222244', linewidth=0.4)
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')

fig.suptitle('Key Extracted Features — Distribution by Class', color='white', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train/Test Split (session-aware, no data leakage) ────────────
X_train, X_test, y_train, y_test = split_dataset(feature_df, test_size=0.2, seed=42)

print('Train/Test split (session-aware, GroupShuffleSplit):')
print(f'  X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'  X_test : {X_test.shape}    y_test : {y_test.shape}')
print(f'\nTrain label distribution:')
print(y_train.value_counts().sort_index().rename(NAMES))
print(f'\nTest label distribution:')
print(y_test.value_counts().sort_index().rename(NAMES))

In [ ]:
print('\n✅ Feature engineering complete.')
print('   Saved to: data/processed/features.csv')
print(f'   {X_train.shape[1]} features per window')
print('\n   Next: notebooks/04_train_model.ipynb')